# Fine Tune LLM Open AI GPT 4o

In [6]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

# Load the variables from .env
load_dotenv()

# Grab the key from the environment manually
# Or replace os.getenv with "actual-key-string" if .env is failing
api_key = os.getenv("OPENAI_API_KEY")

# Pass the key EXPLICITLY to the client
client = OpenAI(api_key=api_key)

file_path = "../data/fine_tuning_llm_training_orlando_mkt_data.jsonl"

# Upload and create the job
with open(file_path, "rb") as f:
    response = client.files.create(file=f, purpose="fine-tune")

file_id = response.id
print(f"File uploaded successfully. File ID: {file_id}")

# Launch the Job
job = client.fine_tuning.jobs.create(
    training_file=file_id, 
    model="gpt-4o-mini-2024-07-18"
)

print(f"Job created! Job ID: {job.id}")
print(f"Current Status: {job.status}")

File uploaded successfully. File ID: file-5dA3coUVVNuz5DprfrcN3R
Job created! Job ID: ftjob-Yoy0vRCJqnhmadGNCNGY6G9P
Current Status: validating_files


In [7]:
# Check status in a new cell later:
client.fine_tuning.jobs.retrieve(job.id)

FineTuningJob(id='ftjob-Yoy0vRCJqnhmadGNCNGY6G9P', created_at=1771194988, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=1.8, n_epochs=3), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-rAXc0BFpoclW2OHUOgbKZFOi', result_files=[], seed=732350159, status='validating_files', trained_tokens=None, training_file='file-5dA3coUVVNuz5DprfrcN3R', validation_file=None, estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=1.8, n_epochs=3))), user_provided_suffix=None, usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None)

In [16]:
# Check the status of specific job
job_status = client.fine_tuning.jobs.retrieve("ftjob-Yoy0vRCJqnhmadGNCNGY6G9P")

print(f"Status: {job_status.status}")

if job_status.status == 'succeeded':
    print(f" Success! Your Model ID is: {job_status.fine_tuned_model}")
elif job_status.status == 'failed':
    print(f" mError: {job_status.error}")
else:
    print(" Still working... Check back in a few minutes.")

Status: succeeded
 Success! Your Model ID is: ft:gpt-4o-mini-2024-07-18:personal::D9f85khV


### Model Registry to serve MCP

In [17]:
# Save this info to a small JSON file so future MCP server can read it

metadata = {
    "model_id": job_status.fine_tuned_model, # Only works after status is 'succeeded'
    "system_prompt": "You are an Orlando real estate expert.",
    "project": "In-House Orlando Market Data Expert",
    "date_trained": "2026-02-15"
}

with open("../output/orlando_mkt_data_expert_metadata.json", "w") as f:
    json.dump(metadata, f)

print("Metadata saved! You can now close this notebook and use the JSON in your MCP server.")

Metadata saved! You can now close this notebook and use the JSON in your MCP server.


In [18]:
metadata

{'model_id': 'ft:gpt-4o-mini-2024-07-18:personal::D9f85khV',
 'system_prompt': 'You are an Orlando real estate expert.',
 'project': 'In-House Orlando Market Data Expert',
 'date_trained': '2026-02-15'}